# Ch14. Neural Networks
**Forecasting: Principles & Practice (Python Edition)**
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

The cells below follow the slide deck *Ch14. Neural networks* (updated, 36 slides) and are meant to be run from top to bottom: later cells reuse objects created in earlier ones.

*Notes:* the AutoNHITS cell needs `ray[tune]`; neural-network results vary slightly with package version and hardware.

In [ ]:
%pip install -q neuralforecast utilsforecast matplotlib "ray[tune]"


In [ ]:
BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"  # datasets in this repository (data/)


## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import MLP, NHITS, RNN, NBEATS, BiTCN
from neuralforecast.auto import AutoNHITS
from neuralforecast.losses.pytorch import MAE, MSE, DistributionLoss
from neuralforecast.utils import AirPassengersPanel
plt.rcParams.update({"figure.figsize": (7, 3.5)})


## [Slide 9] 14.2 MLP for Air Passengers

In [ ]:
test_mask = AirPassengersPanel["ds"] >= "1960"
Y_train_df = AirPassengersPanel[~test_mask]
Y_test_df  = (AirPassengersPanel[test_mask]
              .reset_index(drop=True))

model = MLP(
    h=12,            # forecast horizon
    input_size=24,   # 24 months of history
    scaler_type="robust",
)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()


## [Slide 15] 14.3 MLP, NHITS and RNN Example

In [ ]:
models = [
    MLP(h=12,   input_size=24, scaler_type="robust"),
    NHITS(h=12, input_size=24, scaler_type="robust"),
    RNN(h=12,   input_size=24, scaler_type="robust"),
]
nf = NeuralForecast(models=models, freq="M")
nf.fit(df=Y_train_df)
# columns: unique_id, ds, MLP, NHITS, RNN
forecasts = nf.predict()


## [Slide 16] 14.3 MLP, NHITS and RNN Example — Plot Code

In [ ]:
# last 4 years only
df = AirPassengersPanel[AirPassengersPanel["ds"] >= "1957"]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 4.6))
for ax, uid in zip(axes, ["Airline1", "Airline2"]):
    d = df[df["unique_id"] == uid]
    f = forecasts[forecasts["unique_id"] == uid]
    ax.plot(d["ds"], d["y"], color="black", label="y")
    for m in ["MLP", "NHITS", "RNN"]:
        ax.plot(f["ds"], f[m], label=m)
    # dotted line at the forecast origin (1959-12)
    ax.axvline(d["ds"].iloc[-13], color="gray", ls=":")
    ax.set_title(uid, loc="left")
axes[-1].set_xlabel("Month [1M]")
fig.supylabel("Passengers")
fig.suptitle("Number of passengers for different airlines")
fig.legend(*axes[0].get_legend_handles_labels(),
           loc="lower center", ncol=4)


## [Slide 18] 14.4 Scaling the Data

In [ ]:
# Poor performance without scaling:
model = MLP(h=12, input_size=24, scaler_type="identity")


## [Slide 19] 14.5 Optimisation Objectives

In [ ]:
model = NHITS(h=12, input_size=24,
              loss=MSE(), scaler_type="robust")


## [Slide 20] 14.5 Probabilistic Optimisation Objectives

In [ ]:
model = NHITS(h=12, input_size=24, scaler_type="robust",
              loss=DistributionLoss(distribution="Normal"))
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
# columns: NHITS (mean), NHITS-median, NHITS-lo-90, ..., NHITS-hi-90
forecasts = nf.predict()

df = AirPassengersPanel[AirPassengersPanel["ds"] >= "1957"]
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 4.6))
for ax, uid in zip(axes, ["Airline1", "Airline2"]):
    d = df[df["unique_id"] == uid]
    f = forecasts[forecasts["unique_id"] == uid]
    ax.plot(d["ds"], d["y"], color="black", label="y")
    for lv, a in [(90, .2), (80, .35)]:
        ax.fill_between(f["ds"], f[f"NHITS-lo-{lv}"], f[f"NHITS-hi-{lv}"],
                        color="C0", alpha=a, label=f"{lv}% PI")
    ax.plot(f["ds"], f["NHITS"], color="C0", label="NHITS (mean)")
    ax.set_title(uid, loc="left")
fig.supylabel("Passengers")
fig.legend(*axes[0].get_legend_handles_labels(),
           loc="lower center", ncol=4)


## [Slide 22] 14.6 Exogenous Variables

In [ ]:
df        = pd.read_csv(BASE + "EPF_FR_BE.csv",
                         parse_dates=["ds"])
static_df = pd.read_csv(BASE + "EPF_FR_BE_static.csv")
futr_df   = pd.read_csv(BASE + "EPF_FR_BE_futr.csv",
                         parse_dates=["ds"])


## [Slide 24] 14.6 BiTCN with Exogenous Variables

In [ ]:
horizon = 24   # day-ahead hourly forecast

model = BiTCN(
    h=horizon,
    input_size=5 * horizon,
    futr_exog_list=["gen_forecast", "week_day"],
    hist_exog_list=["system_load"],
    stat_exog_list=["market_0", "market_1"],
    scaler_type="robust",
    max_steps=300,
)
nf = NeuralForecast(models=[model], freq="H")
nf.fit(df=df, static_df=static_df)
forecasts = nf.predict(futr_df=futr_df)


## [Slide 27] 14.7 Hyperparameter Optimisation — Code

In [ ]:
from ray import tune
from neuralforecast.auto import AutoNHITS  # not in .models

nhits_config = {
    **AutoNHITS.get_default_config(h=12, backend="ray"),
    "random_seed": tune.randint(1, 10),
    "n_pool_kernel_size": tune.choice(
        [[2, 2, 2], [16, 8, 1]]
    ),
    "max_steps": tune.choice([100]),
}
model = AutoNHITS(h=12, num_samples=10,
                  config=nhits_config)
nf = NeuralForecast(models=[model], freq="M")
nf.fit(df=Y_train_df)
forecasts = nf.predict()


## [Slide 30] 14.7 Cross-Validation for Neural Networks

In [ ]:
nf = NeuralForecast(
    models=[NHITS(h=12, input_size=24,
                  loss=MAE(), max_steps=500)],
    freq="M"  # month-end dates
)
# refit=False (default): train once, predict all windows
cv_df = nf.cross_validation(
    df=Y_train_df,
    n_windows=3,
    step_size=12,
)
